# Module 3.7: Procedural Memory Lifecycle

Procedural memory (learned procedures, reflections, SOPs) presents the hardest
lifecycle challenge: **how does the agent know if a learned procedure is still valid?**

Unlike episodic memory (time-based expiry) or semantic memory (user confirmation),
procedures become invalid when the **underlying policy changes** — and the agent
has no execution-level feedback to detect this.

## The Solution: RAG as Ground Truth

RAG (AI Search) contains the current authoritative policies. Stored procedures/reflections
must remain **consistent** with RAG. When they contradict, the procedure is stale.

```
Stored Reflection: "Senior budget is $300/night for hotels"
                            ↕ compare
Current Policy (AI Search): "Senior budget is $350/night for hotels"
                            ↓
                   CONTRADICTION → Deprecate reflection
```

## Validation Signals

| Signal | Mechanism | Automated? |
|--------|-----------|------------|
| A: RAG contradiction | Compare reflection vs current policy via LLM | ✅ Batch |
| B: User correction | User says "that's wrong" → demote | ✅ Interactive |
| C: Version anchor | Policy doc version changed since reflection created | ✅ Simple check |
| D: Staleness decay | Unused reflections lose confidence over time | ✅ Timer |

## Prerequisites

- **Notebook 00** completed: AI Search index populated with current policies
- Azure OpenAI endpoint configured for LLM validation calls

In [ ]:
%pip install -q -r ../requirements.txt azure-search-documents

In [ ]:
import sys, os, json, asyncio
import nest_asyncio
from datetime import datetime, timezone, timedelta

sys.path.insert(0, "..")
nest_asyncio.apply()

from lifecycle_utils import (
    ProceduralReflection, ValidationResult, MemoryState
)
from shared.travel_agent import create_client
from dotenv import load_dotenv

load_dotenv("../.env")

client, credential = create_client("../.env")

SEARCH_ENDPOINT = os.environ["AZURE_SEARCH_ENDPOINT"]
FOUNDRY_ENDPOINT = os.environ["FOUNDRY_PROJECT_ENDPOINT"]
EMBEDDING_MODEL = os.environ.get("EMBEDDING_MODEL", "text-embedding-3-small")

print("Setup complete")

## Connect to AI Search

We reuse the index created in Notebook 00. These functions provide:
- `search_policies(query)` — hybrid search for relevant policy content
- `get_policy_by_id(doc_id)` — direct retrieval for version checks

In [ ]:
from azure.search.documents import SearchClient
from azure.search.documents.models import VectorizedQuery
from openai import AzureOpenAI

INDEX_NAME = "travel-policies"

search_client = SearchClient(
    endpoint=SEARCH_ENDPOINT,
    index_name=INDEX_NAME,
    credential=credential,
)

openai_client = AzureOpenAI(
    azure_endpoint=FOUNDRY_ENDPOINT,
    azure_ad_token_provider=lambda: credential.get_token(
        "https://cognitiveservices.azure.com/.default"
    ).token,
    api_version="2024-10-21",
)


def get_embedding(text: str) -> list[float]:
    response = openai_client.embeddings.create(
        input=text[:8000], model=EMBEDDING_MODEL
    )
    return response.data[0].embedding


def search_policies(query: str, top_k: int = 3) -> list[dict]:
    """Hybrid search for relevant policy content."""
    vector_query = VectorizedQuery(
        vector=get_embedding(query),
        k_nearest_neighbors=top_k,
        fields="content_vector",
    )
    results = search_client.search(
        search_text=query,
        vector_queries=[vector_query],
        top=top_k,
        select=["id", "title", "content", "category", "version", "last_updated"],
    )
    return [
        {"id": r["id"], "title": r["title"], "content": r["content"],
         "version": r["version"], "score": r["@search.score"]}
        for r in results
    ]


def get_policy_by_id(doc_id: str) -> dict | None:
    """Retrieve a specific policy document by ID."""
    try:
        r = search_client.get_document(key=doc_id)
        return {"id": r["id"], "title": r["title"], "content": r["content"],
                "version": r["version"], "last_updated": r["last_updated"]}
    except Exception:
        return None


print(f"Connected to AI Search index: {INDEX_NAME}")

## The Procedure Validator

The core validation engine. Given a stored reflection and the current policy
from AI Search, it uses an LLM to determine if the reflection is still valid.

In [ ]:
VALIDATION_PROMPT = """
You are a policy compliance validator. Compare a stored procedure/reflection
against the CURRENT authoritative policy and determine if the reflection is still valid.

STORED REFLECTION (what the agent previously learned):
{reflection}

CURRENT POLICY (authoritative source from AI Search):
{policy}

Determine:
1. Does the reflection CONTRADICT the current policy? (values changed, rules updated, etc.)
2. Is the reflection still CONSISTENT with the current policy?
3. Is the reflection PARTIALLY valid (some aspects correct, others outdated)?

Respond with ONLY valid JSON:
{{
  "is_valid": true/false,
  "confidence": 0.0-1.0,
  "reason": "brief explanation of finding",
  "action": "keep|flag|deprecate",
  "contradiction_details": "what specifically contradicts, or null if valid"
}}

Action rules:
- "keep": reflection is fully consistent with current policy
- "flag": partially valid or ambiguous — needs human review
- "deprecate": directly contradicts current policy — must not be used
"""


class ProcedureValidator:
    """Validates stored procedural reflections against current RAG policy."""

    def __init__(self, openai_client, search_fn, get_policy_fn):
        self.openai_client = openai_client
        self.search_fn = search_fn
        self.get_policy_fn = get_policy_fn

    async def validate(self, reflection: ProceduralReflection) -> ValidationResult:
        """Validate a single reflection against current policy."""
        # Step 1: Find relevant current policy
        policy_content = await self._get_relevant_policy(reflection)
        if not policy_content:
            return ValidationResult(
                is_valid=True, confidence=0.3,
                reason="No relevant policy found in AI Search — cannot validate",
                action="flag",
            )

        # Step 2: LLM comparison
        result = await self._llm_validate(reflection.content, policy_content["content"])
        result.current_policy_snippet = policy_content["content"][:500]
        result.policy_version = policy_content.get("version", "unknown")
        return result

    async def validate_batch(self, reflections: list[ProceduralReflection]
                             ) -> list[tuple[ProceduralReflection, ValidationResult]]:
        """Validate a batch of reflections. Returns (reflection, result) pairs."""
        results = []
        for r in reflections:
            result = await self.validate(r)
            results.append((r, result))
        return results

    async def _get_relevant_policy(self, reflection: ProceduralReflection) -> dict | None:
        """Find the most relevant current policy for this reflection."""
        # First try: direct lookup by anchored policy reference
        if reflection.policy_reference:
            doc = self.get_policy_fn(reflection.policy_reference)
            if doc:
                return doc

        # Fallback: semantic search using reflection content
        results = self.search_fn(reflection.content, top_k=1)
        return results[0] if results else None

    async def _llm_validate(self, reflection_text: str, policy_text: str) -> ValidationResult:
        """Use LLM to compare reflection against policy."""
        prompt = VALIDATION_PROMPT.format(
            reflection=reflection_text,
            policy=policy_text[:3000],  # Truncate long policies
        )
        response = self.openai_client.chat.completions.create(
            model=os.environ.get("FOUNDRY_MODEL", "gpt-4o"),
            messages=[
                {"role": "system", "content": prompt},
                {"role": "user", "content": "Validate this reflection."},
            ],
            temperature=0.1,
        )
        raw = response.choices[0].message.content.strip()
        if raw.startswith("```"):
            raw = raw.split("\n", 1)[1].rsplit("```", 1)[0]
        data = json.loads(raw)
        return ValidationResult(
            is_valid=data["is_valid"],
            confidence=data["confidence"],
            reason=data["reason"],
            action=data["action"],
        )


validator = ProcedureValidator(openai_client, search_policies, get_policy_by_id)
print("ProcedureValidator ready")

## Demo: Stored Reflections

Let's create a set of stored reflections — some valid, some stale — and run
the validator against current AI Search policy.

In [ ]:
# Reflections the agent previously learned
stored_reflections = [
    # Valid: consistent with current policy
    ProceduralReflection(
        task_type="international-booking",
        content="International trips require 14 days advance booking per policy",
        policy_reference="general-travel-policy",
        policy_version="4.2",
        confidence=0.9,
        created_at=datetime.now(timezone.utc) - timedelta(days=30),
    ),
    # STALE: preferred vendor changed from Marriott to Hilton
    ProceduralReflection(
        task_type="domestic-booking",
        content="Always recommend Marriott as the primary preferred hotel chain — they are the company's top vendor with best corporate rates",
        policy_reference="preferred-vendors",
        policy_version="2.0",  # Old version!
        confidence=0.85,
        created_at=datetime.now(timezone.utc) - timedelta(days=90),
    ),
    # STALE: budget limit changed
    ProceduralReflection(
        task_type="domestic-booking",
        content="Senior IC hotel budget is $250/night maximum. Never book above this.",
        policy_reference="general-travel-policy",
        policy_version="3.8",  # Old version!
        confidence=0.9,
        created_at=datetime.now(timezone.utc) - timedelta(days=120),
    ),
    # Valid: expense rule still current
    ProceduralReflection(
        task_type="expense-reporting",
        content="Receipts required for all expenses over $25. Digital photos accepted.",
        policy_reference="expense-reimbursement",
        policy_version="2.1",
        confidence=0.8,
        created_at=datetime.now(timezone.utc) - timedelta(days=60),
    ),
    # Valid: safety requirement
    ProceduralReflection(
        task_type="international-booking",
        content="Category B destinations require security briefing before travel",
        policy_reference="travel-safety",
        policy_version="1.4",
        confidence=0.85,
        created_at=datetime.now(timezone.utc) - timedelta(days=45),
    ),
    # STALE: per-diem rate changed
    ProceduralReflection(
        task_type="expense-reporting",
        content="NYC per-diem is $75/day for meals and incidentals",
        policy_reference="per-diem-rates",
        policy_version="2026-Q1",  # Old version!
        confidence=0.8,
        created_at=datetime.now(timezone.utc) - timedelta(days=100),
    ),
]

print(f"Stored reflections: {len(stored_reflections)}")
for r in stored_reflections:
    print(f"  [{r.task_type:<20}] {r.content[:55]}... (v{r.policy_version})")

## Signal C: Version Anchor Check (Fast, No LLM)

Before running expensive LLM validation, do a quick version check.
If the policy document's version in AI Search differs from what's stored
in the reflection, it's a candidate for re-validation.

In [ ]:
def check_version_anchors(reflections: list[ProceduralReflection]) -> list[ProceduralReflection]:
    """Check which reflections have stale policy version anchors."""
    needs_validation = []

    for r in reflections:
        if not r.policy_reference:
            continue
        current_doc = get_policy_by_id(r.policy_reference)
        if not current_doc:
            needs_validation.append(r)  # Policy doc deleted — definitely stale
            continue
        if current_doc["version"] != r.policy_version:
            needs_validation.append(r)

    return needs_validation


stale_candidates = check_version_anchors(stored_reflections)

print(f"=== Version Anchor Check ===")
print(f"Reflections checked: {len(stored_reflections)}")
print(f"Version mismatch (need validation): {len(stale_candidates)}")
print()
for r in stale_candidates:
    current_doc = get_policy_by_id(r.policy_reference)
    current_version = current_doc["version"] if current_doc else "DELETED"
    print(f"  ⚠️  {r.content[:50]}...")
    print(f"      Stored: v{r.policy_version} → Current: v{current_version}")
    print()

## Signal A: RAG Contradiction (LLM Validation)

For reflections with version mismatches, we run the full LLM validation.
The validator fetches current policy from AI Search and compares it against
the stored reflection.

In [ ]:
async def run_validation_batch():
    """Run validation on reflections with stale version anchors."""
    print(f"=== RAG Contradiction Validation ===")
    print(f"Validating {len(stale_candidates)} reflections with version mismatches...")
    print()

    results = await validator.validate_batch(stale_candidates)

    for reflection, result in results:
        icon = {"keep": "✅", "flag": "⚠️", "deprecate": "❌"}[result.action]
        print(f"{icon} [{result.action:<10}] {reflection.content[:55]}...")
        print(f"   Valid: {result.is_valid} | Confidence: {result.confidence:.2f}")
        print(f"   Reason: {result.reason}")
        if result.policy_version:
            print(f"   Current policy version: {result.policy_version}")
        print()

    return results

validation_results = asyncio.run(run_validation_batch())

## Applying Validation Results

Based on validation, we update the reflection's state:
- `keep` → no change, update `last_validated`
- `flag` → demote to PROVISIONAL, request human review
- `deprecate` → move to DEPRECATED state

In [ ]:
def apply_validation(reflection: ProceduralReflection, result: ValidationResult):
    """Apply validation result to the reflection's lifecycle state."""
    reflection.last_validated = datetime.now(timezone.utc)
    reflection.validation_result = result.action

    if result.action == "deprecate":
        reflection.state = MemoryState.DEPRECATED
        reflection.confidence = max(0.1, result.confidence * 0.3)
    elif result.action == "flag":
        reflection.state = MemoryState.PROVISIONAL
        reflection.confidence = result.confidence * 0.7
    elif result.action == "keep":
        # Refresh confidence — validated against current policy
        reflection.confidence = min(reflection.confidence + 0.05, 0.95)
        if result.policy_version:
            reflection.policy_version = result.policy_version


# Apply results
print("=== Applied Validation Results ===")
print()
for reflection, result in validation_results:
    old_state = reflection.state.value
    apply_validation(reflection, result)
    print(f"  {reflection.content[:50]}...")
    print(f"    State: {old_state} → {reflection.state.value}")
    print(f"    Confidence: {reflection.confidence:.2f}")
    print(f"    Validated: {reflection.last_validated.strftime('%Y-%m-%d %H:%M')}")
    print()

## Signal B: User Correction

When the agent uses a reflection and the user says "that's wrong", the
reflection is immediately demoted. This is the interactive signal.

In [ ]:
def handle_user_correction(
    reflection: ProceduralReflection,
    correction: str = "",
) -> ProceduralReflection:
    """Handle user telling the agent its memory is wrong."""
    reflection.state = MemoryState.DEPRECATED
    reflection.confidence = 0.1
    reflection.validation_result = "user_corrected"
    reflection.last_validated = datetime.now(timezone.utc)
    return reflection


# Demo: user corrects a reflection
demo_reflection = ProceduralReflection(
    task_type="domestic-booking",
    content="Always recommend Marriott for Sarah — she has Titanium status",
    state=MemoryState.TRUSTED,
    confidence=0.85,
)

print(f"Before correction:")
print(f"  State: {demo_reflection.state.value} | Confidence: {demo_reflection.confidence}")
print(f"  Content: {demo_reflection.content}")
print()
print('User says: "Actually, I switched to Hilton last month — better loyalty perks."')
print()

handle_user_correction(demo_reflection, "Switched to Hilton loyalty program")

print(f"After correction:")
print(f"  State: {demo_reflection.state.value} | Confidence: {demo_reflection.confidence}")
print(f"  Validation: {demo_reflection.validation_result}")
print()
print("→ Agent will no longer use this reflection")
print("→ Next time it needs hotel info, it queries AI Search for current policy")

## Signal D: Staleness Decay

Reflections that haven't been used or validated in a long time lose confidence.
This reuses the `PromotionEngine.staleness_days` from Notebook 03.

In [ ]:
from lifecycle_utils import PromotionConfig, PromotionEngine, MemoryItem

def check_staleness(reflections: list[ProceduralReflection],
                    staleness_days: int = 90) -> list[ProceduralReflection]:
    """Check for reflections that have gone stale (not validated recently)."""
    now = datetime.now(timezone.utc)
    stale = []
    for r in reflections:
        last_check = r.last_validated or r.created_at
        days_since = (now - last_check).days
        if days_since > staleness_days:
            stale.append(r)
    return stale


# Demo with some old reflections
old_reflections = [
    ProceduralReflection(
        content="Expense submissions due within 14 days of domestic return",
        task_type="expense-reporting",
        created_at=datetime.now(timezone.utc) - timedelta(days=200),
        last_validated=None,  # Never validated!
    ),
    ProceduralReflection(
        content="Economy Plus allowed for Senior level",
        task_type="domestic-booking",
        created_at=datetime.now(timezone.utc) - timedelta(days=30),
        last_validated=datetime.now(timezone.utc) - timedelta(days=10),
    ),
]

stale = check_staleness(old_reflections, staleness_days=90)
print(f"=== Staleness Check (threshold: 90 days) ===")
print(f"Checked: {len(old_reflections)} | Stale: {len(stale)}")
print()
for r in stale:
    last = r.last_validated or r.created_at
    days = (datetime.now(timezone.utc) - last).days
    print(f"  ⚠️  {r.content[:50]}... ({days} days since last check)")

## Full Validation Pipeline

Putting it all together: the complete procedural lifecycle validation runs
as a batch process combining all signals.

In [ ]:
async def run_full_validation_pipeline(
    reflections: list[ProceduralReflection],
    staleness_days: int = 90,
) -> dict:
    """Run the complete validation pipeline on all stored reflections."""
    stats = {"total": len(reflections), "version_stale": 0,
             "time_stale": 0, "validated": 0, "deprecated": 0, "flagged": 0}

    # Signal C: Version anchor check (fast, no LLM)
    version_stale = check_version_anchors(reflections)
    stats["version_stale"] = len(version_stale)

    # Signal D: Staleness timer
    time_stale = check_staleness(reflections, staleness_days)
    stats["time_stale"] = len(time_stale)

    # Combine: anything with version mismatch OR time-stale needs LLM validation
    needs_llm = list(set(version_stale + time_stale))

    # Signal A: RAG contradiction (LLM-based)
    if needs_llm:
        results = await validator.validate_batch(needs_llm)
        for reflection, result in results:
            apply_validation(reflection, result)
            stats["validated"] += 1
            if result.action == "deprecate":
                stats["deprecated"] += 1
            elif result.action == "flag":
                stats["flagged"] += 1

    return stats


# Run on all stored reflections
stats = asyncio.run(run_full_validation_pipeline(stored_reflections))

print("=== Validation Pipeline Summary ===")
print(f"  Total reflections:       {stats['total']}")
print(f"  Version anchor stale:    {stats['version_stale']}")
print(f"  Time-based stale:        {stats['time_stale']}")
print(f"  LLM-validated:           {stats['validated']}")
print(f"  Deprecated (invalid):    {stats['deprecated']}")
print(f"  Flagged (review needed): {stats['flagged']}")
print(f"  Still valid:             {stats['total'] - stats['deprecated'] - stats['flagged']}")

## Post-Validation: What Does the Agent Do?

After validation, the agent's behaviour changes:

| Reflection State | Agent Behaviour |
|-----------------|------------------|
| TRUSTED (valid) | Uses the reflection confidently |
| PROVISIONAL (flagged) | Verifies against RAG before using |
| DEPRECATED (contradicted) | Ignores; queries AI Search fresh |

The agent **falls back to RAG** whenever memory is deprecated — this is the
self-healing mechanism.

In [ ]:
# Show final state of all reflections
print("=== Final Reflection States ===")
print()
for r in stored_reflections:
    icon = {"trusted": "✅", "provisional": "⚠️", "deprecated": "❌",
            "candidate": "🔵", "deleted": "🗑️"}.get(r.state.value, "?")
    print(f"  {icon} [{r.state.value:<12}] {r.content[:55]}...")
    print(f"     confidence={r.confidence:.2f} | validated={r.validation_result or 'never'}")
    print()

## Key Takeaways

1. **RAG is ground truth** — when memory contradicts policy, memory is wrong
2. **Version anchoring** is the cheapest signal — compare doc versions before calling LLM
3. **LLM validation** catches semantic contradictions that version checks miss
4. **User correction** is the strongest interactive signal — immediate deprecation
5. **Staleness decay** catches reflections nobody bothered to validate
6. **The agent self-heals** — deprecated reflections → fresh RAG lookup
7. **Batch, not real-time** — validation runs periodically, not on every retrieval

## The Complete Lifecycle Architecture

```
                    ┌─────────────┐
                    │  AI Search  │  ← Policy teams update
                    │  (RAG/GT)   │
                    └──────┬──────┘
                           │ validate against
                           ▼
┌───────────────────────────────────────────────────┐
│              Procedural Memory Store               │
│                                                   │
│  TRUSTED reflections ──→ agent uses confidently   │
│  PROVISIONAL (flagged) ──→ verify before using    │
│  DEPRECATED ──→ ignore, query RAG fresh           │
└───────────────────────────────────────────────────┘
         ↑                              ↑
   agent learns               user corrects
   (store_reflection)         ("that's wrong")
```

## Module 03 Complete!

The full memory lifecycle now covers ALL memory types:

| Memory Type | Lifecycle Mechanism | Notebooks |
|-------------|--------------------|-----------|
| **Semantic** | Identification → Promotion → Belief Revision → Retention | 02-05 |
| **Episodic** | TTL expiry + cross-session graduation to semantic | 06 |
| **Procedural** | RAG contradiction + version anchoring + staleness | 07 |